# Chapter 7: Directing a voice

This notebook runs every piece of code in the chapter, from the saved results, in order. Read the chapter alongside it: [Directing a voice](https://books.sugiv.fyi/voice_applications/directing-a-voice.html).


In [ ]:
# Run this cell first. In Google Colab it fetches the companion repo; anywhere else, run the notebook
# from a clone of it. Either way it moves into the book's folder and installs what the chapter needs.
import os
import pathlib
import sys

if "google.colab" in sys.modules and not pathlib.Path("/content/voices-of-the-future").exists():
    !git clone -q --depth 1 https://github.com/sugix/voices-of-the-future /content/voices-of-the-future
candidates = [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, pathlib.Path("/content/voices-of-the-future")]
root = next(p for p in candidates if (p / "voice" / "sim").is_dir())
os.chdir(root / "voice")
%pip install -q -r ../requirements/results.txt

## The same voice for good news and bad

In [ ]:
import json
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display

sys.path.insert(0, "sim")
from wav import load  # Chapter 2's WAV reader

synth = {}
for part in ["kokoro", "kyutai", "fish1", "fish2", "fish2ref"]:
    synth |= json.load(open(f"data/ch7/synth_{part}.json"))["clips"]
heard = json.load(open("data/ch7/asr.json"))  # Parakeet-TDT's transcript of every clip
judged = json.load(open("data/ch7/judge.json"))  # pitch, pauses, loudness, emotion, voice similarity


def play(name):
    x, sr = load(f"data/ch7/listen/{name}.wav")
    display(Audio(x, rate=sr))


def words_per_second(key):
    """Words in the script (without any tag) per second of actual speech."""
    text = re.sub(r"[\(\[].*?[\)\]]", "", synth[key]["text"])
    return len(text.split()) / judged[key]["speech_s"]


def describe(key):
    j = judged[key]
    return (f"{j['f0_median']:4.0f} Hz, pitch range {j['f0_range_semitones']:4.1f} semitones, "
            f"{words_per_second(key):.1f} words/s, {j['loudness_db']:6.1f} dB")


for scene in ["good", "sorry"]:
    key = f"kokoro/{scene}_plain_clean_0"
    print(f"{scene:6s} {describe(key)}   {synth[key]['text']}")
play("kokoro_sorry_plain_clean_0")

## Pace: the dial that works

In [ ]:
line = "readback"
sweep = sorted((synth[k]["speed"], k) for k in synth
               if synth[k]["model"] == "kokoro" and synth[k]["scene"] == line and synth[k]["variant"] != "commas")
print(synth[sweep[0][1]]["text"], "\n")
print("speed  words/s  pause (s)  length (s)")
for speed, k in sweep:
    print(f"{speed:5.1f}  {words_per_second(k):7.2f}  {judged[k]['pause_s']:9.2f}  {synth[k]['seconds']:10.2f}")

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot([s for s, _ in sweep], [words_per_second(k) for _, k in sweep], "o-")
ax.set(xlabel="Kokoro speed setting", ylabel="words per second of speech",
       title="Kokoro's speed is a real dial")
plt.tight_layout()
plt.show()

for k in ["kokoro_readback_plain_clean_0", "kokoro_readback_directed_clean_0"]:
    print(k.split("_")[2], "speed", synth[k.replace("_r", "/r", 1)]["speed"])
    play(k)

In [ ]:
def mean(model, scene, variant, measure, ref="clean"):
    keys = [k for k, r in synth.items() if r["model"] == model and r["scene"] == scene
            and r["variant"] == variant and r.get("ref", "clean") in (ref, "ex04")]
    return np.mean([measure(k) for k in keys])


ASKS = [("fish1", "aside", "(in a hurry tone)"), ("fish2", "aside", "[quickly]"),
        ("fish2", "readback", "[slowly and clearly]"), ("kyutai", "aside", "'fast' voice"),
        ("kyutai", "readback", "'enunciated' voice")]
NAMES = {"fish1": "Fish S1-mini", "fish2": "Fish S2-Pro", "kyutai": "Kyutai TTS 1.6B", "kokoro": "Kokoro-82M"}
print(f"{'':16s} {'asked for':22s} {'plain':>8s} {'directed':>9s}  words/s")
for model, scene, ask in ASKS:
    plain, directed = (mean(model, scene, v, words_per_second) for v in ["plain", "directed"])
    print(f"{NAMES[model]:16s} {ask:22s} {plain:8.2f} {directed:9.2f}  {100 * (directed / plain - 1):+4.0f}%")

## Emotion comes from the reference

In [ ]:
rows = []
for model, ref, label in [("fish2ref", "ex04_default", "S2-Pro, neutral reference"),
                          ("fish2ref", "ex04_happy", "S2-Pro, happy reference"),
                          ("fish2ref", "ex04_sad", "S2-Pro, sad reference"),
                          ("kyutai", "default", "Kyutai, default voice"),
                          ("kyutai", "sad-sympathetic", "Kyutai, sad-sympathetic voice")]:
    for variant in ["plain", "directed"]:
        if model == "kyutai":  # Kyutai's "direction" is the style of its reference voice
            if (variant == "directed") != (ref != "default"):
                continue
            keys = [k for k, r in synth.items() if r["model"] == "kyutai" and r["scene"] == "sorry"
                    and r.get("style") == ref]
        else:
            keys = [k for k, r in synth.items() if r["model"] == model and r["scene"] == "sorry"
                    and r["ref"] == ref and r["variant"] == variant]
        f0 = np.mean([judged[k]["f0_median"] for k in keys])
        rng = np.mean([judged[k]["f0_range_semitones"] for k in keys])
        tag = "with tag" if variant == "directed" and model != "kyutai" else ""
        rows.append((f"{label} {tag}".strip(), f0, rng))

print(f"{'reading the apology':42s} {'pitch':>7s} {'range':>8s}")
for label, f0, rng in rows:
    print(f"{label:42s} {f0:5.0f} Hz {rng:5.1f} st")

fig, ax = plt.subplots(figsize=(7, 3.4))
fish_rows = {label: (f0, rng) for label, f0, rng in rows if label.startswith("S2-Pro")}
for mood, color in [("neutral", "#7f7f7f"), ("happy", "#4a7fb0"), ("sad", "#c0504d")]:
    plain = fish_rows[f"S2-Pro, {mood} reference"]
    tagged = fish_rows[f"S2-Pro, {mood} reference with tag"]
    ax.scatter(*plain, s=50, color=color)
    ax.annotate("", xy=tagged, xytext=plain, arrowprops=dict(arrowstyle="->", color=color))
    ax.annotate(f"{mood} reference", plain, textcoords="offset points", xytext=(-10, 8), fontsize=8, color=color)
ax.set(xlabel="median pitch (Hz)", ylabel="pitch range (semitones)", xlim=(190, 275), ylim=(2, 11),
       title="S2-Pro's apology: the dot is the reference, the arrow is the tag")
plt.tight_layout()
plt.show()

for name in ["fish2ref_sorry_plain_ex04_happy_0", "fish2ref_sorry_plain_ex04_sad_0", "kyutai_sorry_directed_clean_0"]:
    print(name.replace("fish2ref_", "S2-Pro ").replace("_0", "").replace("_", " "))
    play(name)

## Saying names right

In [ ]:
print(f"{'':16s} {'name':8s} {'as written':26s} {'with help':26s}")
for model in ["kokoro", "fish1", "fish2", "kyutai"]:
    helped = "phonemes" if model == "kokoro" else "respelled"
    for name in ["Okafor", "Aoife", "Siobhan", "Domane"]:
        plain, fixed = (heard[f"{model}/name_{name}_{v}_0"] for v in ["spelled", helped])
        print(f"{NAMES[model]:16s} {name:8s} {plain[:26]:26s} {fixed[:26]:26s}")
print()
for v in ["spelled", "phonemes"]:
    print("Kokoro, Okafor", v, ":", synth[f"kokoro/name_Okafor_{v}_0"]["phonemes"])
    play(f"kokoro_name_Okafor_{v}_0")

## A clone copies the recording, not just the voice

In [ ]:
print(f"{'':14s} {'reference':10s} {'energy above 4 kHz':>19s} {'similarity to the speaker':>27s}")
for model in ["fish1", "fish2"]:
    for ref in ["clean", "phone"]:
        keys = [k for k, r in synth.items() if r["model"] == model and r.get("ref") == ref and r["variant"] == "plain"
                and not r["scene"].startswith("name_")]
        above = np.mean([judged[k]["above_4k"] for k in keys])
        sim = np.mean([judged[k]["sim_p228"] for k in keys])
        print(f"{NAMES[model]:14s} {ref:10s} {100 * above:18.2f}% {sim:27.3f}")
print("\nthe reference, over the phone line:")
play("ref_p228_phone")
print("S2-Pro cloned from it:")
play("fish2_good_plain_phone_0")

## Rerun the models

Everything above used the saved results. The cells below rerun the models that produced them, on a GPU (in Colab:
Runtime, then Change runtime type, then T4 GPU). Each step builds the software it needs in its own environment the
first time, which takes a few minutes, then runs. The first cell below sets up a two-item check that everything
works, on a scratch copy of the folder, so the saved results stay as they are. Delete its lines for a full rerun,
which overwrites the saved results with yours. On GPUs without bfloat16, such as the T4, the scripts use float16
automatically.

In [ ]:
# A two-item check that every step runs, on a scratch copy so the saved results stay untouched.
# Delete these lines for a full rerun, which overwrites the saved results with your own.
import os, shutil, tempfile
os.environ["VOTF_QUICK"] = "2"
scratch = os.path.join(tempfile.gettempdir(), "votf-quick")
shutil.copytree(".", scratch, dirs_exist_ok=True)
os.chdir(scratch)

The reference voice from Chapter 6.

In [ ]:
!bash tools/stack.sh nemo
!~/.votf/envs/nemo/bin/python sim/capture_ch6.py --part voice

In [ ]:
!bash tools/stack.sh tts
!~/.votf/envs/tts/bin/python sim/capture_ch7.py --part kokoro

In [ ]:
!bash tools/stack.sh moshi
!~/.votf/envs/moshi/bin/python sim/capture_ch7.py --part kyutai

Fish S1-mini: accept its licence at huggingface.co/fishaudio/s1-mini and log in first.

In [ ]:
!bash tools/stack.sh fish1
!~/.votf/envs/fish1/bin/python sim/capture_ch7.py --part fish1

Fish S2-Pro needs about 24 GB of GPU memory; it doesn't fit a 16 GB T4. Skip it there.

In [ ]:
!bash tools/stack.sh fish2
!~/.votf/envs/fish2/bin/python sim/capture_ch7.py --part fish2

Speaker ex04 in three moods: the references for cloning S2-Pro.

In [ ]:
!bash tools/stack.sh nemo
!~/.votf/envs/nemo/bin/python sim/capture_ch7.py --part refs

S2-Pro cloned from each mood: 24 GB again.

In [ ]:
!bash tools/stack.sh fish2
!~/.votf/envs/fish2/bin/python sim/capture_ch7.py --part fish2ref

In [ ]:
!bash tools/stack.sh nemo
!~/.votf/envs/nemo/bin/python sim/capture_ch7.py --part asr

In [ ]:
!bash tools/stack.sh hf
!~/.votf/envs/hf/bin/python sim/capture_ch7.py --part judge

In [ ]:
!bash tools/stack.sh nemo
!~/.votf/envs/nemo/bin/python sim/capture_ch7.py --part listen